In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/113-1-dl-app-midterm-re-test/reuters21578_subset_test.csv
/kaggle/input/113-1-dl-app-midterm-re-test/reuters21578_subset_train.csv
/kaggle/input/113-1-dl-app-midterm-re-test/reuters21578_subset_test_random_ans.csv


In [2]:
# Cell 1: 導入所需的函式庫
import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
import re
from tqdm import tqdm

In [3]:
# Cell 2: 讀取資料
print("讀取數據...")
try:
    train_df = pd.read_csv('/kaggle/input/113-1-dl-app-midterm-re-test/reuters21578_subset_train.csv', encoding='utf-8')
    test_df = pd.read_csv('/kaggle/input/113-1-dl-app-midterm-re-test/reuters21578_subset_test.csv', encoding='utf-8')
except UnicodeDecodeError:
    train_df = pd.read_csv('/kaggle/input/113-1-dl-app-midterm-re-test/reuters21578_subset_train.csv', encoding='latin1')
    test_df = pd.read_csv('/kaggle/input/113-1-dl-app-midterm-re-test/reuters21578_subset_test.csv', encoding='latin1')


讀取數據...


In [4]:
train_df

,topic_body,topics
0,hr ct vs ct net vs revs ln vs ln nine th hr ct...,earn
1,k mart corp say recent talk sell remain kresge...,acq
2,french sugar producer gh say currently plan wi...,sugar
3,taiwan say plan another round tariff cut possi...,trade
4,hr ct vs ct net ln vs ln revs billion vs ln av...,earn
...,...,...
4448,period end jan hr loss three ct vs loss two ct...,earn
4449,qt ly div eight ct vs seven ct prior payable m...,earn
4450,hr dlr vs dlr net vs reuter,earn
4451,rex com systems corp say agree buy assets post...,acq


In [5]:
test_df

,Id,topic_body
0,1,bank zambia pay foreign exchange arrear three ...
1,2,ban corp inc say board declare initial dividen...
2,3,saudi crude oil output last month fall average...
3,4,gulf states utilities co say condition signifi...
4,5,first data management co inc say complete merg...
...,...,...
2965,2966,american television communications corp say co...
2966,2967,qt ly div ct vs ct prior q tr payable april on...
2967,2968,greyhound corp say sign definitive agreement b...
2968,2969,fruit loom inc say agree sell general battery ...


In [6]:
print("訓練集大小:", len(train_df))
print("測試集大小:", len(test_df))

訓練集大小: 4453
測試集大小: 2970


In [7]:
# Cell 3: 定義預處理函數
def preprocess_text(text):
    # 轉換為小寫
    text = text.lower()
    
    # 移除HTML標籤
    text = re.sub(r'<.*?>', '', text)
    
    # 只保留英文字母和空格
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    return text

In [8]:
# Cell 4: 預處理文本
print("預處理文本...")
train_texts = [preprocess_text(topic_body) for topic_body in tqdm(train_df['topic_body'])]
test_texts = [preprocess_text(topic_body) for topic_body in tqdm(test_df['topic_body'])]

預處理文本...


100%|██████████| 2970/2970 [00:00<00:00, 121431.82it/s]


In [9]:
# Cell 5: 設置 tokenizer
print("創建詞彙表...")
max_words = 10000  # 最大詞彙量
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train_texts + test_texts)

創建詞彙表...


In [10]:
# 轉換文本為序列
train_sequences = tokenizer.texts_to_sequences(train_texts)
test_sequences = tokenizer.texts_to_sequences(test_texts)

In [11]:
# Cell 6: 填充序列
maxlen = 200  # 最大序列長度
train_padded = pad_sequences(train_sequences, maxlen=maxlen)
test_padded = pad_sequences(test_sequences, maxlen=maxlen)

In [12]:
# Cell 7: 創建簡單的 Word2Vec 模型
embedding_dim = 100  # 詞向量維度

In [13]:
model = Sequential([
    Embedding(max_words, embedding_dim, input_length=maxlen),
    GlobalAveragePooling1D()
])

/opt/conda/lib/python3.10/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
# Cell 8: 獲取文本向量表示
print("生成文本向量...")
train_vectors = model.predict(train_padded)
test_vectors = model.predict(test_padded)

生成文本向量...
140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [15]:
# Cell 9: 訓練分類器
print("訓練分類器...")
le = LabelEncoder()
train_labels = le.fit_transform(train_df['topics'])

訓練分類器...


In [16]:
classifier = LogisticRegression(max_iter=1000)
classifier.fit(train_vectors, train_labels)

LogisticRegression(max_iter=1000)

In [17]:
# Cell 10: 進行預測與輸出結果
print("進行預測...")
predictions = classifier.predict(test_vectors)
predictions_labels = le.inverse_transform(predictions)

進行預測...


In [18]:
# 創建提交文件
submit_df = pd.DataFrame({
    'Id': range(len(predictions_labels)),
    'Prediction': predictions_labels
})

In [23]:
submit_df

,Id,Prediction
0,0,earn
1,1,earn
2,2,acq
3,3,acq
4,4,earn
...,...,...
2965,2965,earn
2966,2966,earn
2967,2967,earn
2968,2968,earn


In [19]:
# 保存預測結果
submit_df.to_csv('113-1 DL APP Midterm Re-test Reusters.csv', index=False)
print("預測結果已保存到 '113-1 DL APP Midterm Re-test Reusters.csv.csv'")

預測結果已保存到 '113-1 DL APP Midterm Re-test Reusters.csv.csv'


In [20]:
# Cell 11: 顯示結果統計
print("\n預測結果統計：")
print(submit_df['Prediction'].value_counts())
print("\n前5個預測結果：")
print(submit_df.head())


預測結果統計：
Prediction
earn    2447
acq      523
Name: count, dtype: int64

前5個預測結果：
   Id Prediction
0   0       earn
1   1       earn
2   2        acq
3   3        acq
4   4       earn


In [21]:
# Cell 12 (可選): 詞彙表分析
print("\n詞彙表信息:")
print(f"詞彙表大小: {len(tokenizer.word_index)}")
print(f"向量維度: {embedding_dim}")


詞彙表信息:
詞彙表大小: 7127
向量維度: 100


In [22]:
# Cell 13 (可選): 顯示一些詞的索引
print("\n一些常見詞的索引:")
sample_words = ['movie', 'good', 'bad', 'excellent']
for word in sample_words:
    if word in tokenizer.word_index:
        print(f"{word}: {tokenizer.word_index[word]}")
    else:
        print(f"{word} 不在詞彙表中")


一些常見詞的索引:
movie 不在詞彙表中
good: 189
bad: 1272
excellent: 2848
